# ML data check: who could we predict type 2 diabetes for?

**A planning notebook for the machine learning series (7.x), not a class.** Run it once and share the **summary tables** (all counts of 1 to 20 are hidden). It decides how the 7.x classes are built.

**Why this check?** A prediction model must only use data from **before** the outcome. If we "predict" type 2 diabetes from data collected after the diagnosis, the model learns the consequences of diabetes (and its treatment), not the warning signs. So for each kind of data we need its **date**, and then:

| Question | Why it matters |
|---|---|
| Who **has** this data? | the most people we could use |
| Who **already had** type 2 diabetes by then? | they can't be used to predict it: they're left out |
| Who was **at risk** (no diabetes yet, and health records afterwards)? | the people a model learns from |
| How many **developed** type 2 diabetes afterwards, and over how many years? | too few new cases, and a model can't learn |

| | |
|---|---|
| ⏱️ **Run time** | about 5 minutes |
| 💰 **Cost** | the cohort query (cents to under $1) and one query on surveys, BMI, and Fitbit days (priced first with `run_if_cheap()`, under $2) |
| 💾 **Saves** | nothing |

> ⚠️ **Keep All of Us data in the Workbench.** Share only the summary tables. Clear outputs before committing.

## 1. Setup and the cohort

In [ ]:
source("../aou_setup.R")

cohort <- run_query(str_glue(read_file("../sql/t2d_cohort.sql"), AGE_DATE = "2025-01-01")) %>%
  mutate(person_id = as.character(person_id),
         t2d_date  = as.Date(first_t2d_date),
         last_ehr  = as.Date(last_ehr_date),
         death     = as.Date(death_date))

cohort %>% count(cohort) %>% hide_small_counts()

**🔍 Read the output:** cases, controls, and excluded (type 1 or unclear), as in every series.

## 2. When each person's data was collected

One query finds, for everyone:

- **Enrollment:** the date of their **first survey** (The Basics, answered when they join)
- **BMI at enrollment:** a plausible BMI measured from a year before enrollment to 30 days after (the program's physical measurements are taken around enrollment)
- **Fitbit:** their first day of Fitbit activity data (the daily summary table, not the minute-level ones)

`run_if_cheap()` prices it first and stops if it would cost more than the limit.

In [ ]:
dates_sql <- str_glue("
WITH enroll AS (
  SELECT person_id, MIN(DATE(survey_datetime)) AS enroll_date
  FROM `{CURRENT_DATASET}.ds_survey`
  GROUP BY person_id
),
bmi AS (
  SELECT DISTINCT m.person_id
  FROM `{CURRENT_DATASET}.measurement` AS m
  JOIN enroll AS e ON m.person_id = e.person_id
  WHERE m.measurement_concept_id = 3038553                 -- BMI
    AND m.value_as_number BETWEEN 12 AND 80                -- plausible values (2.1)
    AND m.measurement_date BETWEEN DATE_SUB(e.enroll_date, INTERVAL 1 YEAR)
                               AND DATE_ADD(e.enroll_date, INTERVAL 30 DAY)
),
fitbit AS (
  SELECT person_id, MIN(date) AS first_fitbit
  FROM `{CURRENT_DATASET}.activity_summary`
  GROUP BY person_id
)
SELECT e.person_id, e.enroll_date,
       bmi.person_id IS NOT NULL AS bmi_at_enroll,
       f.first_fitbit
FROM enroll AS e
LEFT JOIN bmi    ON e.person_id = bmi.person_id
LEFT JOIN fitbit AS f ON e.person_id = f.person_id")

data_dates <- run_if_cheap(dates_sql, max_dollars = 2) %>%
  mutate(person_id = as.character(person_id))
last_query_cost()

data_dates %>%
  summarise(people          = n(),
            with_bmi        = sum(bmi_at_enroll),
            with_fitbit     = sum(!is.na(first_fitbit)),
            enroll_from     = year(min(enroll_date)),
            enroll_median   = year(median(enroll_date)),
            enroll_to       = year(max(enroll_date)))

**🔍 Read the output:** the cost, then how many people answered a survey, had a BMI at enrollment, and have Fitbit data; and the years enrollment spans (years only, over hundreds of thousands of people).

## 3. Proteomics: is there a sample date?

The proteomics files are in the Workbench, not BigQuery. The **manifest** lists each sample. We need to know **when** the blood was drawn: before or after diagnosis. This cell prints the manifest's **column names** (safe to share) and looks for a date.

In [ ]:
PROT_DIR <- file.path(aou_data_dir("v9"), "multiomics", "proteomics")
manifest <- read_tsv(file.path(PROT_DIR, "manifest.tsv"), col_types = cols(.default = "c"))

names(manifest)

date_cols <- names(manifest)[str_detect(names(manifest), regex("date|time|collect|draw", ignore_case = TRUE))]
date_cols

**🔍 Read the output:** the column names, then any that look like dates. **Please share both lists.**

- **A date column?** The next cell uses the first one as each person's proteomics date. (If it isn't the collection date, tell me and change `PROT_DATE_COL`.)
- **None?** The next cell uses **enrollment** instead, and says so. Biospecimens are usually collected around enrollment, but that's an assumption we'd have to state in 7.3.

In [ ]:
PROT_DATE_COL <- if (length(date_cols) > 0) date_cols[1] else NA_character_

prot <- manifest %>%
  transmute(person_id = research_id,
            prot_date = if (!is.na(PROT_DATE_COL)) as.Date(substr(.data[[PROT_DATE_COL]], 1, 10)) else as.Date(NA)) %>%
  group_by(person_id) %>%
  summarise(prot_date = suppressWarnings(min(prot_date, na.rm = TRUE)), .groups = "drop") %>%
  mutate(prot_date = if_else(is.infinite(prot_date), as.Date(NA), prot_date))

c(people_with_proteomics = nrow(prot),
  with_a_date            = sum(!is.na(prot$prot_date)),
  date_column_used       = PROT_DATE_COL)

if (any(!is.na(prot$prot_date))) {
  prot %>% summarise(from = year(min(prot_date, na.rm = TRUE)), median = year(median(prot_date, na.rm = TRUE)),
                     to = year(max(prot_date, na.rm = TRUE)))
}

**🔍 Read the output:** how many people have proteomics, how many with a usable date, and which column was used; then the range of years, if there's a date.

## 4. Genomes

Genomes don't need a date: DNA doesn't change after a diagnosis, so a genetic variant is always "before". We only need to know **who** has one.

In [ ]:
psam_file <- file.path(aou_data_dir("v9"), "wgs", "short_read", "snpindel", "acaf_threshold", "pgen",
                       "acaf_threshold.chr10.psam")
genome_ids <- read_tsv(psam_file, col_types = cols(.default = "c"))[["#IID"]]
length(genome_ids)

## 5. Who could each kind of data predict for?

`at_risk_summary()` takes everyone with a given kind of data and its **time zero** (when it was collected), and counts:

| Column | Means |
|---|---|
| `people` | in the cohort with this data |
| `diabetes_already` | type 2 diabetes on or before time zero: can't be used |
| `at_risk` | not excluded, no type 2 diabetes yet, and health records **after** time zero |
| `new_t2d` | of those at risk, how many developed type 2 diabetes afterwards |
| `median_years` | median follow-up after time zero, among those at risk |

Counts of 1 to 20 are hidden (and the median, if `at_risk` is hidden). Follow-up ends at the earlier of the last EHR visit, death, and the data cutoff (2025-01-01).

In [ ]:
end_of_data <- as.Date("2025-01-01")

at_risk_summary <- function(people, label) {
  people %>%
    mutate(already  = cohort == "case" & !is.na(t2d_date) & t2d_date <= time_zero,
           end      = pmin(last_ehr, death, end_of_data, na.rm = TRUE),
           at_risk  = cohort != "excluded" & !already & end > time_zero,
           new_t2d  = at_risk & cohort == "case" & t2d_date > time_zero,
           years    = as.numeric(end - time_zero) / 365.25) %>%
    summarise(median_years     = round(median(years[at_risk]), 1),   # first: the next lines replace at_risk with its count
              data             = label,
              people           = n(),
              diabetes_already = sum(already),
              at_risk          = sum(at_risk),
              new_t2d          = sum(new_t2d)) %>%
    relocate(median_years, .after = new_t2d)
}

base <- cohort %>%
  select(person_id, cohort, t2d_date, last_ehr, death) %>%
  inner_join(data_dates, by = "person_id") %>%
  left_join(prot, by = "person_id") %>%
  mutate(has_prot   = person_id %in% prot$person_id,
         has_genome = person_id %in% genome_ids,
         prot_zero  = coalesce(prot_date, enroll_date))

overlap <- bind_rows(
  at_risk_summary(base %>% mutate(time_zero = enroll_date), "Surveys (enrollment)"),
  at_risk_summary(base %>% filter(bmi_at_enroll) %>% mutate(time_zero = enroll_date), "+ BMI at enrollment"),
  at_risk_summary(base %>% filter(bmi_at_enroll, has_genome) %>% mutate(time_zero = enroll_date), "+ BMI + genome"),
  at_risk_summary(base %>% filter(!is.na(first_fitbit)) %>% mutate(time_zero = first_fitbit), "Fitbit (first day)"),
  at_risk_summary(base %>% filter(bmi_at_enroll, !is.na(first_fitbit)) %>%
                    mutate(time_zero = pmax(enroll_date, first_fitbit)), "Surveys + BMI + Fitbit"),
  at_risk_summary(base %>% filter(has_prot) %>% mutate(time_zero = prot_zero), "Proteomics"),
  at_risk_summary(base %>% filter(has_prot, bmi_at_enroll, has_genome) %>%
                    mutate(time_zero = pmax(prot_zero, enroll_date)), "Proteomics + BMI + genome"),
  at_risk_summary(base %>% filter(has_prot, !is.na(first_fitbit)) %>%
                    mutate(time_zero = pmax(prot_zero, first_fitbit)), "Proteomics + Fitbit")
) %>%
  mutate(across(c(people, diabetes_already, at_risk, new_t2d),
                ~ if_else(.x >= 1 & .x <= 20, NA_integer_, .x)),   # hide counts of 1 to 20
         median_years = if_else(is.na(at_risk), NA_real_, median_years))   # and medians of so few people

overlap

**🔍 Read the output:** one row per kind of data (or combination). **Please share this table**: it's what decides the 7.x designs.

- `new_t2d` is the number that matters most. A model needs **hundreds** of new cases to learn from, and to test on. Under about 100, a combination is too small for the main analysis.
- `diabetes_already`: for proteomics, a large number means most samples were taken **after** diagnosis. Then 7.3 can't predict new diabetes from proteins, and uses a related question instead.
- `median_years`: short follow-up means fewer new cases, and predictions over a short horizon only.

## 6. Follow-up: when was the blood drawn?

The proteomics manifest has **no date**, so section 5 used **enrollment** as the proteomics time zero. That's only right if the blood was drawn around enrollment.

The genome files have a table with a **`biosample_collection_date`** for each genome sample ("the date the collection site finalized the order, which is generally close to … the actual time of collection", from *How the All of Us genomic and multi-omic data are organized*). Most people with proteomics also have a genome. If their genome blood draw is close to enrollment, that supports the assumption.

This cell finds the table (it looks for the column in every file in the genome `aux/qc` folder) and reads only two columns.

In [ ]:
QC_DIR <- file.path(aou_data_dir("v9"), "wgs", "short_read", "snpindel", "aux", "qc")
list.files(QC_DIR)

qc_files  <- list.files(QC_DIR, pattern = "\\.tsv$", full.names = TRUE)
date_file <- qc_files[map_lgl(qc_files, ~ "biosample_collection_date" %in%
                                names(read_tsv(.x, n_max = 0, show_col_types = FALSE)))]
basename(date_file)

**🔍 Read the output:** the files in the folder, then the one with the collection date. **Please share both.** If the second is empty (`character(0)`), the table is somewhere else: tell me, and we'll look for it.

In [ ]:
draws <- read_tsv(date_file[1], col_types = cols(.default = "c"),
                  col_select = c(research_id, biosample_collection_date)) %>%
  transmute(person_id = research_id,
            draw_date = as.Date(parse_date_time(biosample_collection_date, orders = c("mdy", "ymd"))))

prot_draws <- base %>%
  filter(has_prot) %>%
  inner_join(draws, by = "person_id") %>%
  mutate(days_after_enrollment = as.numeric(draw_date - enroll_date))

# How close is the blood draw to enrollment? (people with proteomics and a genome date)
prot_draws %>%
  summarise(people                = n(),
            median_days           = median(days_after_enrollment, na.rm = TRUE),
            pct_within_30_days    = round(100 * mean(abs(days_after_enrollment) <= 30, na.rm = TRUE), 1),
            pct_within_1_year     = round(100 * mean(abs(days_after_enrollment) <= 365, na.rm = TRUE), 1),
            pct_over_1_year_after = round(100 * mean(days_after_enrollment > 365, na.rm = TRUE), 1))

**🔍 Read the output:** for people with proteomics, how many days after enrollment their genome blood was drawn. **Please share this row.**

- **Most within 30 days?** Then the blood was drawn at the enrollment visit, and enrollment is a fair time zero for proteins.
- **Many over a year after?** Then some "new" cases were diagnosed **before** the blood was drawn. The next cell counts them.

In [ ]:
# The proteomics rows again, with the genome blood-draw date as time zero
bind_rows(
  at_risk_summary(prot_draws %>% mutate(time_zero = enroll_date), "Proteomics: enrollment"),
  at_risk_summary(prot_draws %>% mutate(time_zero = pmax(enroll_date, draw_date)),
                  "Proteomics: later of enrollment and blood draw")
) %>%
  mutate(across(c(people, diabetes_already, at_risk, new_t2d),
                ~ if_else(.x >= 1 & .x <= 20, NA_integer_, .x)),
         median_years = if_else(is.na(at_risk), NA_real_, median_years))

**🔍 Read the output:** the same people, with two time zeros. **Please share this table.** If `new_t2d` barely changes, the timing assumption doesn't matter much, and 7.3 can use enrollment. If it drops a lot, 7.3 uses the blood-draw date.

## Clean up

Nothing was saved. **Clear outputs** before committing: **Edit → Clear Outputs of All Cells**, then save.